# Deep Research Agent

This notebook demonstrates the deep research agent: an iterative loop that decomposes
a question, searches multiple sources, detects gaps and conflicts, and synthesizes
a markdown report with structured references.

## Imports and setup

In [ ]:
from IPython.display import Markdown, display

from agentic_patterns.core.agents.research import (
    DeepResearchAgent,
    SearchSourcePerplexity,
)

## Create a Perplexity search source

Reads the API key from `config.yaml` (section `search.perplexity`).

In [ ]:
import os
import re
import yaml
from agentic_patterns.core.config.config import MAIN_PROJECT_DIR

config_path = MAIN_PROJECT_DIR / "config.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)

search_cfg = cfg["search"]["perplexity"]
api_key = re.sub(
    r"\$\{(\w+)\}",
    lambda m: os.environ.get(m.group(1), m.group(0)),
    search_cfg["api_key"],
)

perplexity_source = SearchSourcePerplexity(
    api_key=api_key,
    model=search_cfg.get("model", "sonar"),
    api_url=search_cfg.get("api_url", "https://api.perplexity.ai"),
)
print(perplexity_source)

## Create and run the research agent

In [ ]:
agent = DeepResearchAgent(sources=[perplexity_source], max_iterations=2)
print(agent)

In [ ]:
report = await agent.run(
    "What are the current best practices for LLM evaluation in production systems?"
)

## Display the report

In [ ]:
display(Markdown(report.content))

## Structured references

In [ ]:
for i, ref in enumerate(report.references, 1):
    print(f"[{i}] ({ref.source_type}) {ref.title}")
    if ref.url:
        print(f"    {ref.url}")
    print(
        f"    {ref.snippet[:120]}..."
        if len(ref.snippet) > 120
        else f"    {ref.snippet}"
    )
    print()

## Multi-source research (web + VectorDB)

If you have a local vector database with relevant documents, you can add it
alongside Perplexity for hybrid web + private knowledge research.

In [ ]:
# Uncomment and adapt to use a local VectorDB source:
#
# from agentic_patterns.core.vectordb.vectordb import get_vector_db
# from agentic_patterns.core.agents.research import SearchSourceVectorDB
#
# vdb = get_vector_db("my_collection")
# vdb_source = SearchSourceVectorDB(vdb)
#
# agent_multi = DeepResearchAgent(sources=[perplexity_source, vdb_source])
# report_multi = await agent_multi.run("How does our internal API handle rate limiting?")
# display(Markdown(report_multi.content))